# 阵列无关 SELD：Kaggle 2×T4 流程

本 notebook 演示：安装 → 获取 PSELDNets 与预训练权重 → 数据仿真/预处理 → DDP 训练（自动续训）→ 评测（含 array swap）→ ONNX 导出。

**会话设置**：Accelerator 选 `GPU T4 x2`，打开 Internet。Kaggle 单次会话 12h，训练脚本会在 `train.max_hours`（默认 11.3h）前保存断点退出；
下个会话把上次的 Notebook 输出挂为 Input，并设置 `train.resume_from=/kaggle/input/<上次输出>/ckpt/<exp>/last.pt` 即可续训。

In [ ]:
!nvidia-smi
import os
# 方式一：把 array_agnostic_seld.zip 作为 Kaggle Dataset 上传后解压；方式二：git clone 你自己的仓库
ZIP = '/kaggle/input/array-agnostic-seld/array_agnostic_seld.zip'   # TODO: 改成你的数据集路径
if os.path.exists(ZIP):
    !cd /kaggle/working && unzip -q -o {ZIP}
%cd /kaggle/working/array_agnostic_seld
!pip install -q pyroomacoustics h5py soundfile onnx onnxruntime onnxscript huggingface_hub

## 1. PSELDNets 代码与预训练权重（CNN14-Conformer, mACCDOA）

In [ ]:
!git clone -q --depth 1 https://github.com/Jinbo-Hu/PSELDNets third_party/PSELDNets || echo 'already cloned'
from huggingface_hub import hf_hub_download
ck = hf_hub_download('Jinbo-HU/PSELDNets', 'model/mACCDOA-CNN14-Conformer-0.582.ckpt', repo_type='dataset',
                     local_dir='/kaggle/working/ckpts')   # ~800MB
print(ck)
# 若无法联网：把权重作为 Kaggle Dataset 挂载，并修改 configs/base.yaml 中 model.backbone.ckpt_path

## 2. 数据仿真（pyroomacoustics）

`sources.csv` 格式：`filename,class_idx,class_name`，`class_idx` 应映射到 STARSS23 的 13 类（0..12）。
可以用 FSD50K 子集（按 STARSS23 类别挑选）构建；这里先用合成 dummy 源跑通。

In [ ]:
DATA = '/kaggle/working/data'
USE_DUMMY = True   # TODO: 换成真实 FSD50K 子集时设为 False，并设置 SRC_CSV
if USE_DUMMY:
    !python -m sim.dummy_sources --out {DATA}/dummy_sources --n-per-class 40
    SRC_CSV = f'{DATA}/dummy_sources/sources.csv'
else:
    SRC_CSV = '/kaggle/input/fsd50k-starss-subset/sources.csv'
N_TRAIN, N_VAL = 2000, 200   # 10s 场景；4 核 CPU 上约 0.3~1 s/场景（取决于 max_order）
!python -m sim.simulate --source-csv {SRC_CSV} --out {DATA}/sim_train --n-scenes {N_TRAIN} --duration 10 --max-order 10 --workers 4
!python -m sim.simulate --source-csv {SRC_CSV} --out {DATA}/sim_val --n-scenes {N_VAL} --duration 10 --max-order 10 --workers 4 --seed 1 --prefix val_
# 未见几何测试集（例如只用 tetra/random）
!python -m sim.simulate --source-csv {SRC_CSV} --out {DATA}/sim_unseen --n-scenes {N_VAL} --duration 10 --max-order 10 --workers 4 --seed 2 --prefix unseen_ --array-types tetra

In [ ]:
!python -m sim.precompute --root {DATA}/sim_train  --out {DATA}/h5/sim_train.h5 --workers 4
!python -m sim.precompute --root {DATA}/sim_val    --out {DATA}/h5/sim_val.h5 --workers 4
!python -m sim.precompute --root {DATA}/sim_unseen --out {DATA}/h5/sim_unseen.h5 --workers 4
!python scripts/check_ls_foa.py --h5 {DATA}/h5/sim_val.h5 --n 5

## 3. STARSS23 (MIC 格式) 准备

In [ ]:
STARSS = '/kaggle/input/starss23'   # TODO: 挂载 STARSS23 (含 mic_dev, foa_dev, metadata_dev)
if os.path.exists(STARSS):
    !python -m sim.prepare_starss23 --starss {STARSS} --out {DATA}/starss23_train --split dev-train
    !python -m sim.prepare_starss23 --starss {STARSS} --out {DATA}/starss23_test --split dev-test
    !python -m sim.precompute --root {DATA}/starss23_train --out {DATA}/h5/starss23_train.h5 --workers 4
    !python -m sim.precompute --root {DATA}/starss23_test  --out {DATA}/h5/starss23_test.h5 --workers 4

## 4. 训练（DDP 2×T4, fp16, 每 epoch 断点 + 自动续训）

In [ ]:
# 阶段零：理想 FOA 上训练教师（PSELDNets 骨干 + 新的 13 类头）
!torchrun --nproc_per_node=2 train.py --config configs/teacher_foa.yaml \
    model.backbone.ckpt_path=/kaggle/working/ckpts/model/mACCDOA-CNN14-Conformer-0.582.ckpt
# 阶段一：仿真预训练前端（同一命令重复执行会自动从 last.pt 续训）；教师/初始化 = 阶段零 best.pt
!torchrun --nproc_per_node=2 train.py --config configs/sim_pretrain.yaml model.backbone.ckpt_path=null

In [ ]:
# 阶段二：STARSS23 MIC 微调
!torchrun --nproc_per_node=2 train.py --config configs/starss23_finetune.yaml \
    model.backbone.ckpt_path=null

In [ ]:
# 阶段三：蒸馏手机端小模型（教师 = 阶段二完整模型）
!torchrun --nproc_per_node=2 train.py --config configs/distill_small.yaml

## 5. 评测（DCASE 风格 + array swap）

In [ ]:
!python eval.py --config configs/sim_pretrain.yaml --ckpt /kaggle/working/ckpt/stage1_sim_pretrain/best.pt \
    --h5 seen={DATA}/h5/sim_val.h5 unseen_geom={DATA}/h5/sim_unseen.h5 --mic-subsets all,2,3,4 \
    --out /kaggle/working/eval_stage1.json model.backbone.ckpt_path=null
if os.path.exists(f'{DATA}/h5/starss23_test.h5'):
    !python eval.py --config configs/starss23_finetune.yaml --ckpt /kaggle/working/ckpt/stage2_starss23_mic/best.pt \
        --h5 starss23={DATA}/h5/starss23_test.h5 --mic-subsets all,3,2 --out /kaggle/working/eval_stage2.json model.backbone.ckpt_path=null

## 6. ONNX 导出（蒸馏小模型）

In [ ]:
!python export/export_onnx.py --config configs/distill_small.yaml --ckpt /kaggle/working/ckpt/stage3_distill_small/best.pt \
    --out /kaggle/working/seld_small.onnx --n-mics 4 teacher.mode=none